In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3
import pickle

try:
    import optbinning
except:
    ! pip install optbinning
    
try:
    import catboost
except:
    ! pip install catboost

#### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20250221-credit-builder-analysis
Task: 01_gen12_predictions


#### Output dir

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# drop
list_cols = [col for col in df.columns if 'tu_pmthx' in col]
list_cols.append('applicationdayofweek__app')
df.drop(list_cols, axis=1, inplace=True)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,1,0,7,NaN,1.311220,1,0,0,0,0.616667
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,0,5,NaN,1.432368,0,0,0,0,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1,0,4,NaN,1.587073,1,0,0,0,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,2,NaN,1.081881,0,1,0,1,0.000000
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,4,NaN,1.371350,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,0,0,NaN,1.198869,1,0,0,0,0.367073
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,0,3,NaN,1.313231,0,0,0,0,NaN
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,1,0,3,NaN,0.991586,1,0,0,0,0.006585


#### Import preprocessor

In [6]:
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in list_str_filename:
    str_bucket_path = f'01_ad/02_model/noPTImodel10/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path,
        str_bucket_path=str_bucket_path,
        str_project='20231010-gen-xii',
    )
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

#### Preprocess data

In [7]:
df = cls_model_preprocessing.transform(df)
# rm
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    os.remove(str_filename)
# show
df

NaN Replacer: 2.0097 sec.


100%|██████████| 3/3 [00:00<00:00, 32.38it/s]

Unable to convert vehiclemodel__app to string, not found in data
Set strings: 0.096939 sec.



/home/ec2-user/SageMaker/20250221_credit_builder_analysis/01_gen12_predictions/preprocessing.py:98: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.replace(dict_replace, inplace=True)


Boolean Replacer: 3.6505 sec.


...

Replace inf and -inf with NaN: 2.9432 sec.
Imputer: 2.7119 sec.
Map term: 0.058713 sec.
Map PTI: 0.06476 sec.


100%|██████████| 9/9 [00:00<00:00, 1076.75it/s]


Round values: 0.013272 sec.
Preprocessing Model: 22.478 sec.


100%|██████████| 2/2 [00:00<00:00, 3511.35it/s]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-dealership_age
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,iowa,franchise,iowa,1.0,...,1,0,0,0,0.616667,2021,1.080027,7,3,9.109589
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,utah,franchise,utah,1.0,...,0,0,0,0,0.000000,2021,1.080027,7,3,2.580822
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,illinois,franchise,illinois,1.0,...,1,0,0,0,0.000000,2021,1.080027,7,3,11.484932
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,michigan,independent,michigan,0.0,...,0,1,0,1,0.000000,2021,1.080027,7,3,6.213699
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,arizona,franchise,arizona,0.0,...,0,1,0,1,0.000000,2021,1.080027,7,3,8.572603
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,northcarolina,independent,northcarolina,1.0,...,0,0,0,0,0.000000,2024,0.940900,11,4,8.235616
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,nevada,franchise,nevada,1.0,...,1,0,0,0,0.367073,2024,0.940900,11,4,0.191781
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,virginia,franchise,virginia,0.0,...,0,0,0,0,0.000000,2024,0.940900,11,4,6.210959
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,alabama,franchise,alabama,1.0,...,1,0,0,0,0.006585,2024,0.940900,10,4,11.345205


#### AD predictions

In [8]:
str_filename = 'final_model.pkl'
str_model = '01_ad'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['ad'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df

/tmp/ipykernel_12242/2335986344.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ad'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-dealership_age,ad
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,iowa,franchise,iowa,1.0,...,0,0,0,0.616667,2021,1.080027,7,3,9.109589,0.268054
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,utah,franchise,utah,1.0,...,0,0,0,0.000000,2021,1.080027,7,3,2.580822,0.421207
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,illinois,franchise,illinois,1.0,...,0,0,0,0.000000,2021,1.080027,7,3,11.484932,0.249006
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,michigan,independent,michigan,0.0,...,1,0,1,0.000000,2021,1.080027,7,3,6.213699,0.514731
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,arizona,franchise,arizona,0.0,...,1,0,1,0.000000,2021,1.080027,7,3,8.572603,0.244784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,northcarolina,independent,northcarolina,1.0,...,0,0,0,0.000000,2024,0.940900,11,4,8.235616,0.610044
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,nevada,franchise,nevada,1.0,...,0,0,0,0.367073,2024,0.940900,11,4,0.191781,0.255144
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,virginia,franchise,virginia,0.0,...,0,0,0,0.000000,2024,0.940900,11,4,6.210959,0.605399
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,alabama,franchise,alabama,1.0,...,0,0,0,0.006585,2024,0.940900,10,4,11.345205,0.292850


#### PD predictions

In [9]:
str_filename = 'final_model.pkl'
str_model = '02_pricing_pd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df

/tmp/ipykernel_12242/1208004230.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-dealership_age,ad,pd
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,iowa,franchise,iowa,1.0,...,0,0,0.616667,2021,1.080027,7,3,9.109589,0.268054,0.052721
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,utah,franchise,utah,1.0,...,0,0,0.000000,2021,1.080027,7,3,2.580822,0.421207,0.242853
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,illinois,franchise,illinois,1.0,...,0,0,0.000000,2021,1.080027,7,3,11.484932,0.249006,0.234714
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,michigan,independent,michigan,0.0,...,0,1,0.000000,2021,1.080027,7,3,6.213699,0.514731,0.309557
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,arizona,franchise,arizona,0.0,...,0,1,0.000000,2021,1.080027,7,3,8.572603,0.244784,0.165092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,northcarolina,independent,northcarolina,1.0,...,0,0,0.000000,2024,0.940900,11,4,8.235616,0.610044,0.207462
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,nevada,franchise,nevada,1.0,...,0,0,0.367073,2024,0.940900,11,4,0.191781,0.255144,0.112541
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,virginia,franchise,virginia,0.0,...,0,0,0.000000,2024,0.940900,11,4,6.210959,0.605399,0.087163
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,alabama,franchise,alabama,1.0,...,0,0,0.006585,2024,0.940900,10,4,11.345205,0.292850,0.161923


#### LGD predictions

In [10]:
str_filename = 'final_model.pkl'
str_model = '03_pricing_lgd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['lgd'] = cls_model_inference.predict(df[list_cols_model])
# show
df

/tmp/ipykernel_12242/2072904958.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lgd'] = cls_model_inference.predict(df[list_cols_model])


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-dealership_age,ad,pd,lgd
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,iowa,franchise,iowa,1.0,...,0,0.616667,2021,1.080027,7,3,9.109589,0.268054,0.052721,0.682529
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,utah,franchise,utah,1.0,...,0,0.000000,2021,1.080027,7,3,2.580822,0.421207,0.242853,0.636985
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,illinois,franchise,illinois,1.0,...,0,0.000000,2021,1.080027,7,3,11.484932,0.249006,0.234714,0.745692
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,michigan,independent,michigan,0.0,...,1,0.000000,2021,1.080027,7,3,6.213699,0.514731,0.309557,0.691098
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1.0,arizona,franchise,arizona,0.0,...,1,0.000000,2021,1.080027,7,3,8.572603,0.244784,0.165092,0.618661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,northcarolina,independent,northcarolina,1.0,...,0,0.000000,2024,0.940900,11,4,8.235616,0.610044,0.207462,0.620092
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,nevada,franchise,nevada,1.0,...,0,0.367073,2024,0.940900,11,4,0.191781,0.255144,0.112541,0.606080
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,virginia,franchise,virginia,0.0,...,0,0.000000,2024,0.940900,11,4,6.210959,0.605399,0.087163,0.650510
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1.0,alabama,franchise,alabama,1.0,...,0,0.006585,2024,0.940900,10,4,11.345205,0.292850,0.161923,0.701278


#### Convert non-numeric to string

In [11]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 2737/2737 [00:10<00:00, 261.22it/s] 


#### Write to s3

In [12]:
%%time

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 30.6 s, sys: 354 ms, total: 31 s
Wall time: 35.2 s
